In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

df.head(3)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes


In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)

label_encoders = {}
for col in df.columns:
    if df[col].dtype == 'object':
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])
        label_encoders[col] = le

if 'customerID' in df.columns:
    df = df.drop('customerID', axis=1)

print("Dane gotowe. Podgląd:")
df.head(3)

Dane gotowe. Podgląd:


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,0,1,0,1,0,1,0,0,2,0,0,0,0,0,1,2,29.85,29.85,0
1,1,0,0,0,34,1,0,0,2,0,2,0,0,0,1,0,3,56.95,1889.50,0
2,1,0,0,0,2,1,0,0,2,2,0,0,0,0,0,1,3,53.85,108.15,1


# A

In [ ]:
df['days_since_last_call'] = np.where(df['Churn'] == 1, 0, np.random.randint(1, 30, len(df)))

X = df.drop('Churn', axis=1)
y = df['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

rf = RandomForestClassifier(n_estimators=50, random_state=42)
rf.fit(X_train, y_train)
acc = accuracy_score(y_test, rf.predict(X_test))

print(f"Accuracy modelu z wyciekiem: {acc:.4f}")

Accuracy modelu z wyciekiem: 1.0000


# B

In [ ]:
# 1. Sprawdzenie korelacji (Pearson)
correlations = df.corr()['Churn'].abs()
high_corr = correlations[(correlations > 0.9) & (correlations.index != 'Churn')].index.tolist()

if high_corr:
    print(f"[FLAGA CZERWONA] Cechy z korelacją > 0.9 z targetem: {high_corr}")
else:
    print("[OK] Brak cech o korelacji > 0.9")

# 2. Feature Importance
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
top_feature_name = importances.index[0]
top_feature_val = importances.values[0]

if top_feature_val > 0.5:
    print(f"[FLAGA CZERWONA] Jedna cecha dominuje (>50% wagi): {top_feature_name} -> {top_feature_val:.2f}")

# 3. Single Feature Prediction
# Trenujemy model używając TYLKO tej jednej, podejrzanej cechy
rf_single = RandomForestClassifier(n_estimators=10, random_state=42)
rf_single.fit(X_train[[top_feature_name]], y_train)
acc_single = accuracy_score(y_test, rf_single.predict(X_test[[top_feature_name]]))

if acc_single > 0.9:
    print(f"[FLAGA CZERWONA] Model oparty tylko na '{top_feature_name}' ma accuracy: {acc_single:.4f}")

[OK] Brak cech o korelacji > 0.9
[FLAGA CZERWONA] Jedna cecha dominuje (>50% wagi): days_since_last_call -> 0.77
[FLAGA CZERWONA] Model oparty tylko na 'days_since_last_call' ma accuracy: 1.0000


# C

* **ZŁE PODEJŚCIE:** `scaler.fit(X)` -> Wyciek informacji ze zbioru testowego (średnia i odchylenie) do treningowego.
* **DOBRE PODEJŚCIE:** `Pipeline` -> Scaler uczy się tylko na `X_train`.

In [ ]:
X_clean = X.drop('days_since_last_call', axis=1)
X_tr_raw, X_te_raw, y_tr, y_te = train_test_split(X_clean, y, test_size=0.3, random_state=42)

# ZŁE PODEJŚCIE (Data Leakage)
scaler_bad = StandardScaler()
# BŁĄD: Skalujemy CAŁE dane przed podziałem (model widzi statystyki zbioru testowego)
X_scaled_all = scaler_bad.fit_transform(X_clean) 
X_tr_bad, X_te_bad, y_tr_bad, y_te_bad = train_test_split(X_scaled_all, y, test_size=0.3, random_state=42)

clf_bad = LogisticRegression(max_iter=1000)
clf_bad.fit(X_tr_bad, y_tr_bad)
acc_bad = accuracy_score(y_te_bad, clf_bad.predict(X_te_bad))

# DOBRE PODEJŚCIE (Pipeline)
# Pipeline gwarantuje, że fit() wykona się tylko na X_train
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000))
])
pipe.fit(X_tr_raw, y_tr)
acc_good = accuracy_score(y_te, pipe.predict(X_te_raw))

print(f"Accuracy (Złe skalowanie - wyciek): {acc_bad:.5f}")
print(f"Accuracy (Dobre - Pipeline):        {acc_good:.5f}")

Accuracy (Złe skalowanie - wyciek): 0.81022
Accuracy (Dobre - Pipeline):        0.81022


Wynik ten sam, ale mogło być inaczej!!!

# D

Lista 5 pytań/testów, które należy wykonać przed wdrożeniem modelu:

1.  Czy każda cecha w zbiorze jest dostępna w momencie podejmowania decyzji? (np. czy nie używamy danych, które generują się dopiero po wystąpieniu zdarzenia churnu?).
2.  Jeśli accuracy > 95% (a szczególnie 99-100%), czy sprawdziłem korelację każdej cechy z targetem i usunąłem te z `|r| > 0.9`? (Korelacja nie implikuje przyczynowości, to może być inny związek)
3.  Czy na pewno usunąłem wszystkie identyfikatory (User ID, Transaction ID)? Czasami ID rosną sekwencyjnie i mogą przypadkowo korelować z targetem.
4.  Czy cały preprocessing (skalowanie, imputacja braków) jest zamknięty w `Pipeline` i fitowany tylko na `X_train`?
5.  Czy `feature_importances_` nie pokazuje jednej cechy, która ma np. 80% wagi, podczas gdy reszta ma śladowe znaczenie?